In [3]:
# ==============================================================================
# PHYSICS-INFORMED HYBRID FRAMEWORK FOR LEAK LOCALISATION
# Verified Data: Van der Walt, Heyns, & Wilke (University of Pretoria, 2021)
# Author: Tevfik Denizhan Muftuoglu
# ==============================================================================

# ==============================================================================
# STEP 0: ENVIRONMENT SETUP & IMPORTS
# ==============================================================================
import importlib, subprocess, sys, os, glob
if importlib.util.find_spec('pywt') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'PyWavelets'])

import numpy as np
import pandas as pd
from scipy import signal, stats
import pywt
import re
from sklearn.linear_model import Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# Journal-quality plot defaults — dpi=500 satisfies MSSP minimum for combination figures
mpl.rcParams.update({
    'font.size': 11,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'savefig.dpi': 500,
    'savefig.bbox': 'tight',
    'legend.fontsize': 10,
    'legend.framealpha': 0.9,
    'legend.edgecolor': 'gray',
})

# ==============================================================================
# STEP 1: WORKSPACE CLEANUP & INTERACTIVE DATA INGESTION
# ==============================================================================
print("--- STEP 1: SYSTEM INITIALIZATION, CLEANUP, AND FILE UPLOAD ---")

for f in glob.glob("*.npy") + glob.glob("*.png"):
    try:
        os.remove(f)
    except OSError:
        pass

print("WORKSPACE CLEANED. PLEASE SELECT YOUR 9 .NPY FILES:")
uploaded = files.upload()

files_found = sorted([f for f in uploaded.keys() if f.endswith('.npy') and 'eak' in f.lower()])
assert len(files_found) == 9, f'CRITICAL ERROR: Expected 9 .npy files, but {len(files_found)} were uploaded.'
print(f"SUCCESS: 9 files successfully loaded. Commencing analysis...\n")

# ==============================================================================
# STEP 2: HYDRAULIC CONSTANTS & BASELINE DEFINITION
# ==============================================================================
FS     = 10_000
C_HW   = 100
D_PIPE = 25e-3
L_PIPE = 65
A_HW   = 10.67 / (C_HW ** 1.852 * D_PIPE ** 4.8704)

def analytic_leak_location(sample):
    P1, Q1, P2, Q2 = sample[0], sample[1]/1000., sample[2], sample[3]/1000.
    num = (P1 - P2) - A_HW * L_PIPE * Q2 ** 1.852
    den = A_HW * (Q1 ** 1.852 - Q2 ** 1.852)
    return num / den if abs(den) > 1e-12 else np.nan

# ==============================================================================
# STEP 3: BOOTSTRAP ANALYSIS
# ==============================================================================
print("--- STEP 2: PERFORMING BOOTSTRAP ANALYSIS ON ANALYTIC BASELINE ---")
rng = np.random.default_rng(0)
N_BOOT, BOOT_SIZE = 500, 5000

bootstrap_results, summary_rows = [], []

for f in files_found:
    d = np.load(f)
    match = re.search(r'(\d+)_(\d+)m', f)
    size_idx = int(match.group(1)) if match else 1
    loc_true  = float(match.group(2)) if match else 0.0
    case_name = re.sub(r'\s*\(\d+\)', '', f).replace('.npy', '')

    P1, Q1 = d[:, 0], d[:, 1]
    summary_rows.append({
        'case': case_name, 'location_m': loc_true, 'size_idx': size_idx,
        'LL_analytic_raw': analytic_leak_location(d.mean(axis=0)),
        'QL': Q1.mean() - d[:, 3].mean()
    })

    preds = []
    for _ in range(N_BOOT):
        idx = rng.integers(0, d.shape[0], size=BOOT_SIZE)
        preds.append(analytic_leak_location(d[idx].mean(axis=0)))
    preds = np.array(preds)
    preds = preds[np.isfinite(preds)]
    ci_lo, ci_hi = np.percentile(preds, [2.5, 97.5])
    bootstrap_results.append({
        'case': case_name, 'location_m': loc_true,
        'boot_mean': preds.mean(), 'ci_lo': ci_lo, 'ci_hi': ci_hi,
    })

summary = pd.DataFrame(summary_rows)
summary['analytic_error'] = summary['LL_analytic_raw'] - summary['location_m']
bootstrap_df = pd.DataFrame(bootstrap_results)

# ==============================================================================
# STEP 4: HYBRID FEATURE EXTRACTION (33 FEATURES)
# ==============================================================================
print("--- STEP 3: EXTRACTING 33-DIMENSIONAL MULTI-SCALE FEATURES ---")

def band_energy(x, fs, lo, hi, nperseg=None):
    x = x - x.mean()
    if nperseg is None: nperseg = min(2048, len(x))
    f, P = signal.welch(x, fs, nperseg=nperseg)
    m = (f >= lo) & (f <= hi)
    return float(np.trapezoid(P[m], f[m]))

def window_features(win):
    P1, Q1, P2, Q2 = win[:, 0], win[:, 1], win[:, 2], win[:, 3]
    dP = P1 - P2
    feats = {
        'P1_mean': P1.mean(), 'Q1_mean': Q1.mean(),
        'P2_mean': P2.mean(), 'Q2_mean': Q2.mean(),
        'dP_mean': dP.mean(), 'QL_mean': Q1.mean() - Q2.mean(),
        'LL_analytic': analytic_leak_location(win.mean(axis=0)),
        'P1_std': P1.std(), 'Q1_std': Q1.std(),
        'P2_std': P2.std(), 'Q2_std': Q2.std(),
        'P1_skew': float(stats.skew(P1)), 'P1_kurt': float(stats.kurtosis(P1)),
        'P2_skew': float(stats.skew(P2)), 'P2_kurt': float(stats.kurtosis(P2)),
        'Q1_skew': float(stats.skew(Q1)), 'Q2_skew': float(stats.skew(Q2)),
        'E_P1_pump':    band_energy(P1, FS, 45,  55),
        'E_P1_100_500': band_energy(P1, FS, 100, 500),
        'E_P1_500_2k':  band_energy(P1, FS, 500, 2000),
        'E_dP_100_500': band_energy(dP, FS, 100, 500),
        'E_dP_500_2k':  band_energy(dP, FS, 500, 2000),
    }

    dx = (dP - dP.mean())[::4]
    coeffs = pywt.wavedec(dx, 'db4', level=4)
    for i, c in enumerate(coeffs):
        feats[f'W_dP_lvl{i}'] = float(np.sum(c ** 2) / len(c))

    fc, Cxy = signal.coherence(P1 - P1.mean(), P2 - P2.mean(), FS, nperseg=min(2048, len(P1)))
    for lo, hi, nm in [(100, 500, 'mid'), (500, 2000, 'high')]:
        mask = (fc >= lo) & (fc <= hi)
        feats[f'coh_{nm}'] = float(Cxy[mask].mean()) if mask.any() else 0.0

    Q1_m3 = Q1 / 1000.0
    res = dP - (A_HW * L_PIPE * Q1_m3 ** 1.852)
    feats['res_mean'] = res.mean()
    feats['res_std']  = res.std()
    feats['res_skew'] = float(stats.skew(res))
    return feats

N_WINDOWS = 30
WIN_LEN   = 150_000 // N_WINDOWS
all_rows  = []

for f in files_found:
    d = np.load(f)
    match = re.search(r'(\d+)_(\d+)m', f)
    loc_true  = float(match.group(2)) if match else 0.0
    case_name = re.sub(r'\s*\(\d+\)', '', f).replace('.npy', '')
    for w in range(N_WINDOWS):
        win  = d[w * WIN_LEN : (w + 1) * WIN_LEN]
        feats = window_features(win)
        feats['case'] = case_name
        feats['loc_true'] = loc_true
        feats['ql_true']  = feats['QL_mean']
        all_rows.append(feats)

df = pd.DataFrame(all_rows)

# ==============================================================================
# STEP 5: LOOCV CROSS-VALIDATION & PLS-k3 MODEL TRAINING
# ==============================================================================
print("--- STEP 4: TRAINING PLS-K3 MODEL VIA LEAVE-ONE-GROUP-OUT CV ---")
feature_cols = [c for c in df.columns if c not in ('case', 'loc_true', 'size_idx', 'ql_true')]
X      = np.nan_to_num(df[feature_cols].values, nan=0, posinf=0, neginf=0)
y_loc  = df['loc_true'].values
groups = df['case'].values

def run_loocv_model(estimator, X, y, groups):
    preds_window = np.zeros(len(y))
    logo = LeaveOneGroupOut()
    for tr, te in logo.split(X, y, groups):
        scaler = StandardScaler().fit(X[tr])
        mdl = estimator().fit(scaler.transform(X[tr]), y[tr])
        preds_window[te] = np.asarray(mdl.predict(scaler.transform(X[te]))).flatten()
    return preds_window

def aggregate_to_case(preds_window, y, groups):
    cases  = np.unique(groups)
    y_true = np.array([y[groups == c][0]           for c in cases])
    y_pred = np.array([preds_window[groups == c].mean() for c in cases])
    return cases, y_true, y_pred

cases, y_true_case, _ = aggregate_to_case(np.zeros(len(y_loc)), y_loc, groups)
analytic_raw = np.array([df.loc[df['case'] == c, 'LL_analytic'].mean() for c in cases])

preds_pls = run_loocv_model(lambda: PLSRegression(n_components=3), X, y_loc, groups)
_, _, y_pred_pls = aggregate_to_case(preds_pls, y_loc, groups)
case_pred_std = np.array([preds_pls[groups == c].std() for c in cases])

def mae(pred, truth): return np.mean(np.abs(pred - truth))
mae_analytic = mae(analytic_raw, y_true_case)
mae_hybrid   = mae(y_pred_pls,   y_true_case)
print(f"   => Analytic Baseline MAE : {mae_analytic:.2f} m")
print(f"   => Hybrid PLS-k3 MAE     : {mae_hybrid:.2f} m  (Improvement: {mae_analytic/mae_hybrid:.1f}x)\n")

# ==============================================================================
# STEP 6: CONFORMAL PREDICTION
# ==============================================================================
print("--- STEP 5: CALCULATING JACKKNIFE+ CONFORMAL INTERVALS ---")

def jackknife_plus_coverage(alpha, y_pred, y_true):
    m = len(y_pred) - 1
    k = min(int(np.ceil((1 - alpha) * (m + 1))), m)
    abs_res = np.abs(y_pred - y_true)
    lo, hi, q_vals = [], [], []
    for i in range(len(y_pred)):
        others = np.delete(abs_res, i)
        q = np.sort(others)[k - 1]
        q_vals.append(q)
        lo.append(y_pred[i] - q)
        hi.append(y_pred[i] + q)
    lo, hi = np.array(lo), np.array(hi)
    covered = (y_true >= lo) & (y_true <= hi)
    return covered.mean(), lo, hi, np.array(q_vals)

_, lo90, hi90, _ = jackknife_plus_coverage(0.1, y_pred_pls, y_true_case)
_, lo80, hi80, _ = jackknife_plus_coverage(0.2, y_pred_pls, y_true_case)

# ==============================================================================
# STEP 7: FIGURE GENERATION
#   - No titles inside figures (captions go in the manuscript)
#   - dpi=500 for all saved figures (MSSP requirement)
#   - FIXED: Fig 3 legends separated and readable
#   - FIXED: Fig 5 pump annotation font enlarged and repositioned
#   - FIXED: Fig 8 legend and coverage stats non-overlapping
# ==============================================================================
print("--- STEP 6: GENERATING, SAVING, AND DOWNLOADING ALL 8 FIGURES ---")
generated_files = []

# ─── FIGURE 1  (FIXED: CI caps thick and visible) ────────────────────────────
fig1, ax1 = plt.subplots(figsize=(9, 5))
y_pos = np.arange(len(bootstrap_df))

for k, r in bootstrap_df.iterrows():
    # Error bar first (so dot renders on top)
    ax1.errorbar(r['boot_mean'], k,
                 xerr=[[r['boot_mean'] - r['ci_lo']], [r['ci_hi'] - r['boot_mean']]],
                 fmt='none',
                 ecolor='#d62728', elinewidth=2.0,
                 capsize=7, capthick=2.0,
                 zorder=2)
    # Mean dot on top
    ax1.scatter(r['boot_mean'], k,
                marker='o', s=70, color='#d62728',
                edgecolor='darkred', linewidth=0.8, zorder=3)
    # True location
    ax1.scatter(r['location_m'], k,
                marker='s', s=90,
                facecolor='#2ca02c', edgecolor='black',
                linewidth=0.8, zorder=4)

ax1.set_yticks(y_pos)
ax1.set_yticklabels([c.replace('Leak ', 'Leak_') for c in bootstrap_df['case']],
                    fontsize=10)
ax1.set_xlabel('Leak location estimate [m]', fontsize=12)
ax1.axvline(0,      color='grey', lw=0.5)
ax1.axvline(L_PIPE, color='grey', lw=0.5, ls=':')
ax1.grid(axis='x', alpha=0.3)
ax1.set_xlim(-70, 80)

custom1 = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#d62728',
           markersize=9, markeredgecolor='darkred',
           label='Bootstrap mean (analytic)'),
    Line2D([0],[0], color='#d62728', lw=2.0,
           label='95% CI (bootstrap)'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor='#2ca02c',
           markersize=9, markeredgecolor='black',
           label='True location'),
]
ax1.legend(handles=custom1, loc='lower right', frameon=True, fontsize=10)
plt.tight_layout()
fig1.savefig('Figure_1.png')
generated_files.append('Figure_1.png')
plt.close(fig1)

# ─── FIGURE 2 ─────────────────────────────────────────────────────────────────
fig2, ax_flow = plt.subplots(figsize=(14, 6))
ax_flow.axis('off')
ax_flow.set_xlim(0, 1.2)
ax_flow.set_ylim(0, 1.1)

nodes = {
    'A':  (0.12, 0.76, "Raw measurements\n$P_1, Q_1, P_2, Q_2$\n15 s @ 10 kHz", "#fff3e0"),
    'B':  (0.12, 0.34, "Segment into\n30 windows × 0.5 s",                        "#e8f5e9"),
    'C1': (0.38, 0.85, "Steady-state physics\nLL_analytic, res_mean, QL",          "#ffe0b2"),
    'C2': (0.38, 0.68, "Spectral (Welch bands)\n45-55 / 100-500 / 500-2k Hz",     "#c8e6c9"),
    'C3': (0.38, 0.51, "Wavelet energies\nDaubechies-4, 5 levels on ΔP",           "#f3e5f5"),
    'C4': (0.38, 0.34, "Higher order stats\nskew, kurt of $P_1, P_2$",             "#e0f7fa"),
    'C5': (0.38, 0.17, "Coherence $\\gamma^2(P_1, P_2)$\nmid-band, high-band",    "#fce4ec"),
    'D':  (0.62, 0.51, "33-dim\nfeature vector\n(per window)",                     "#e3f2fd"),
    'E1': (0.83, 0.70, "Standardise\n(LOOCV-safe)",                                "#f5f5f5"),
    'E2': (0.83, 0.51, "PLS-k3\nrank-3 latent space",                              "#bbdefb"),
    'E3': (0.83, 0.32, "LOOCV\n9 folds (case-level)",                              "#fff3e0"),
    'F':  (1.05, 0.51, "Leak\nlocation\nestimate\n(+ std)",                         "#e8f5e9"),
}
for k, (x, y, text, fc) in nodes.items():
    weight = 'bold' if k in ['D', 'E2', 'F'] else 'normal'
    pad    = 0.8    if k in ['D', 'F']        else 0.6
    ax_flow.text(x, y, text,
                 bbox=dict(boxstyle=f"round,pad={pad}", fc=fc, ec="gray", lw=1),
                 ha='center', va='center', fontsize=9, fontweight=weight, zorder=3)

for src, dst in [('A','B'),('B','C1'),('B','C2'),('B','C3'),('B','C4'),('B','C5'),
                 ('C1','D'),('C2','D'),('C3','D'),('C4','D'),('C5','D'),
                 ('D','E1'),('E1','E2'),('E3','E2'),('E2','F')]:
    ax_flow.annotate("", xy=(nodes[dst][0], nodes[dst][1]),
                     xytext=(nodes[src][0], nodes[src][1]),
                     arrowprops=dict(arrowstyle="->", color="#555555",
                                     shrinkA=45, shrinkB=45, lw=1.2), zorder=1)
ax_flow.text(0.05, 0.05,
             "Features: physics priors + multi-scale signal analysis + higher-order statistics",
             style='italic', color='gray', fontsize=8)
plt.tight_layout()
fig2.savefig('Figure_2.png')
generated_files.append('Figure_2.png')
plt.close(fig2)

# ─── FIGURE 3  (FIXED: panel labels above axes via set_title, no overlap) ─────
fig3, axes = plt.subplots(1, 2, figsize=(12, 5.5))

ax_a = axes[0]
ax_a.plot([0, L_PIPE], [0, L_PIPE], 'k--', lw=0.8, alpha=0.6)
ax_a.scatter(y_true_case, analytic_raw, s=90, alpha=0.9,
             c='#d62728', marker='o', edgecolor='k', linewidth=0.6, zorder=3)
for t, p, c in zip(y_true_case, analytic_raw, cases):
    if abs(p - t) > 20:
        ax_a.annotate(c.replace('Leak ', 'Leak_'), (t, p),
                      xytext=(5, -8), textcoords='offset points', fontsize=8)
ax_a.set_xlabel('True leak location [m]', fontsize=12)
ax_a.set_ylabel('Predicted leak location [m]', fontsize=12)
ax_a.set_xlim(0, L_PIPE); ax_a.set_ylim(-65, 70)
ax_a.axhline(0,      color='grey', lw=0.3)
ax_a.axhline(L_PIPE, color='grey', lw=0.3, ls=':')
ax_a.text(0.97, 0.05, f'MAE = {mae_analytic:.1f} m',
          transform=ax_a.transAxes, fontsize=11, va='bottom', ha='right',
          bbox=dict(boxstyle='round', fc='#fff0f0', ec='#d62728',
                    linewidth=1.5, alpha=0.95))
leg_a = [
    Line2D([0],[0], ls='--', color='k', lw=0.8, label='Ideal (y = x)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#d62728',
           markersize=8, markeredgecolor='k', label='Analytic H-W (raw)'),
]
ax_a.legend(handles=leg_a, loc='upper left', frameon=True, fontsize=10)
# Panel label above axis — completely outside the plot area, no overlap possible
ax_a.set_title('(a)', loc='left', fontsize=13, fontweight='bold', pad=6)

ax_b = axes[1]
ax_b.plot([0, L_PIPE], [0, L_PIPE], 'k--', lw=0.8, alpha=0.6)
ax_b.errorbar(y_true_case, y_pred_pls, yerr=2 * case_pred_std,
              fmt='o', markersize=8, color='#2ca02c', ecolor='#2ca02c',
              alpha=0.85, capsize=4, elinewidth=1.4,
              markeredgecolor='k', markeredgewidth=0.6)
ax_b.set_xlabel('True leak location [m]', fontsize=12)
ax_b.set_ylabel('Predicted leak location [m]', fontsize=12)
ax_b.set_xlim(0, L_PIPE); ax_b.set_ylim(0, L_PIPE)
ax_b.text(0.03, 0.97, f'MAE = {mae_hybrid:.2f} m',
          transform=ax_b.transAxes, fontsize=11, va='top', ha='left',
          bbox=dict(boxstyle='round', fc='#f0fff0', ec='#2ca02c',
                    linewidth=1.5, alpha=0.95))
leg_b = [
    Line2D([0],[0], ls='--', color='k', lw=0.8, label='Ideal (y = x)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#2ca02c',
           markersize=8, markeredgecolor='k', label='Hybrid PLS-k3 (±2σ)'),
]
ax_b.legend(handles=leg_b, loc='lower right', frameon=True, fontsize=10)
# Panel label above axis
ax_b.set_title('(b)', loc='left', fontsize=13, fontweight='bold', pad=6)

plt.tight_layout()
fig3.savefig('Figure_3.png')
generated_files.append('Figure_3.png')
plt.close(fig3)

# ─── FIGURE 4 ─────────────────────────────────────────────────────────────────
err_analytic = np.abs(analytic_raw  - y_true_case)
err_hybrid   = np.abs(y_pred_pls    - y_true_case)
locs         = [15.0, 30.0, 50.0]
grid_analytic = np.full((3, 3), np.nan)
grid_hybrid   = np.full((3, 3), np.nan)

for i in range(len(cases)):
    l_idx = locs.index(y_true_case[i])
    s_val = df.loc[df['case'] == cases[i], 'ql_true'].mean()
    s_idx = 0 if s_val < 0.2 else (1 if s_val < 0.5 else 2)
    grid_analytic[s_idx, l_idx] = err_analytic[i]
    grid_hybrid[s_idx, l_idx]   = err_hybrid[i]

fig4, axes7 = plt.subplots(1, 2, figsize=(12, 4.5))
cmap_r = plt.cm.Reds.copy();   cmap_r.set_bad('lightgray')
cmap_g = plt.cm.Greens.copy(); cmap_g.set_bad('lightgray')

for ax, grid, cmap, label in zip(axes7, [grid_analytic, grid_hybrid],
                                  [cmap_r, cmap_g], ['(a)', '(b)']):
    im = ax.imshow(grid, cmap=cmap, aspect='auto', vmin=0, vmax=10)
    ax.set_xticks([0,1,2]); ax.set_xticklabels([15, 30, 50])
    ax.set_yticks([0,1,2]); ax.set_yticklabels(['Small\n~0.1', 'Medium\n~0.3', 'Large\n~0.6'])
    ax.set_xlabel('True leak location [m]')
    ax.set_ylabel('Leak size [l/s]')
    ax.text(0.5, -0.18, label, transform=ax.transAxes, ha='center', fontsize=12, fontweight='bold')
    for i in range(3):
        for j in range(3):
            val = grid[i, j]
            if np.isnan(val):
                ax.text(j, i, "N/A", ha="center", va="center", color="gray", fontsize=9, style='italic')
            else:
                ax.text(j, i, f"{val:.1f}", ha="center", va="center",
                        color="white" if val > 5 else "black", fontsize=10)
    plt.colorbar(im, ax=ax, label='|error| [m]', fraction=0.046, pad=0.04)

plt.tight_layout()
fig4.savefig('Figure_4.png')
generated_files.append('Figure_4.png')
plt.close(fig4)

# ─── FIGURE 5  (FIXED: pump annotation enlarged and repositioned) ─────────────
sample_file = files_found[0]
d_sample    = np.load(sample_file)
t_3s        = np.arange(30000) / FS

f_welch, psd_p1 = signal.welch(d_sample[:,0]-d_sample[:,0].mean(), FS, nperseg=4096)
_,       psd_q1 = signal.welch(d_sample[:,1]-d_sample[:,1].mean(), FS, nperseg=4096)
_,       psd_p2 = signal.welch(d_sample[:,2]-d_sample[:,2].mean(), FS, nperseg=4096)
_,       psd_q2 = signal.welch(d_sample[:,3]-d_sample[:,3].mean(), FS, nperseg=4096)

channels = [
    (d_sample[:30000, 0], psd_p1, 'P1 [m]',   '#1f77b4'),
    (d_sample[:30000, 1], psd_q1, 'Q1 [l/s]',  '#ff7f0e'),
    (d_sample[:30000, 2], psd_p2, 'P2 [m]',    '#1f77b4'),
    (d_sample[:30000, 3], psd_q2, 'Q2 [l/s]',  '#ff7f0e'),
]

fig5 = plt.figure(figsize=(14, 9))

for i, (ts, psd, ylabel, c) in enumerate(channels):
    # Time-series panel (columns 1-3)
    ax_t = plt.subplot(4, 4, (i*4+1, i*4+3))
    ax_t.plot(t_3s, ts, color=c, lw=0.5)
    ax_t.set_ylabel(ylabel, fontsize=10)
    ax_t.set_xlim(0, 3)
    ax_t.grid(alpha=0.3)
    if i < 3:
        ax_t.set_xticklabels([])
    else:
        ax_t.set_xlabel('Time [s]', fontsize=11)

    # PSD panel (column 4)
    ax_p = plt.subplot(4, 4, i*4+4)
    ax_p.loglog(f_welch, psd, color=c, lw=0.8)
    ax_p.set_ylabel('PSD', fontsize=10)
    ax_p.set_xlim(1, 5000)
    ax_p.grid(alpha=0.3, which='both')

    if i == 0:
        # Pump fundamental annotation — ENLARGED font, clear positioning
        pump_freq = 48.8
        ax_p.axvline(pump_freq, color='#e31a1c', ls='--', lw=1.5, zorder=5)
        # Find a y-position in the middle of the PSD range for the annotation
        mask = (f_welch >= 40) & (f_welch <= 60)
        y_mid = float(np.sqrt(psd[mask].max() * psd[mask].min())) if mask.any() else 1e-5
        ax_p.annotate(
            '48.8 Hz\n(pump)',
            xy=(pump_freq, y_mid),
            xytext=(120, y_mid * 3),           # offset to the right, slightly up
            fontsize=9, color='#e31a1c',
            fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#e31a1c', lw=1.2),
            va='center',
        )

    if i < 3:
        ax_p.set_xticklabels([])
    else:
        ax_p.set_xlabel('Frequency [Hz]', fontsize=11)

plt.tight_layout()
fig5.savefig('Figure_5.png')
generated_files.append('Figure_5.png')
plt.close(fig5)

# ─── FIGURE 6 ─────────────────────────────────────────────────────────────────
scaler_full = StandardScaler().fit(X)
pls_full    = PLSRegression(n_components=3).fit(scaler_full.transform(X), y_loc)
coefs_abs   = np.abs(pls_full.coef_).flatten()

top_idx   = np.argsort(coefs_abs)[-15:]
top_feats = [feature_cols[i] for i in top_idx]
top_vals  = coefs_abs[top_idx]

color_map = []
for ft in top_feats:
    if 'analytic' in ft or 'res' in ft or 'QL' in ft: color_map.append('#ff7f0e')
    elif 'W_dP' in ft: color_map.append('#9467bd')
    elif 'E_' in ft:   color_map.append('#2ca02c')
    else:              color_map.append('#17becf')

fig6, ax5 = plt.subplots(figsize=(10, 6))
ax5.barh(np.arange(len(top_feats)), top_vals, color=color_map,
         edgecolor='k', linewidth=0.5)
ax5.set_yticks(np.arange(len(top_feats)))
ax5.set_yticklabels(top_feats, fontsize=10)
ax5.set_xlabel('Mean |PLS coefficient| across LOOCV folds', fontsize=12)

legend6 = [
    Line2D([0],[0], color='#ff7f0e', lw=6, label='Physics-model derived'),
    Line2D([0],[0], color='#9467bd', lw=6, label='Wavelet multi-scale'),
    Line2D([0],[0], color='#2ca02c', lw=6, label='Spectral band energy'),
    Line2D([0],[0], color='#17becf', lw=6, label='Higher-order statistics'),
]
ax5.legend(handles=legend6, loc='lower right', frameon=True, fontsize=10)
plt.tight_layout()
fig6.savefig('Figure_6.png')
generated_files.append('Figure_6.png')
plt.close(fig6)

# ─── FIGURE 7 ─────────────────────────────────────────────────────────────────
bias_sq     = (y_pred_pls - y_true_case) ** 2
variance    = case_pred_std ** 2
ql_per_case = np.array([df.loc[df['case'] == c, 'ql_true'].mean() for c in cases])
order       = np.lexsort((ql_per_case, y_true_case))

fig7, ax3 = plt.subplots(figsize=(9, 4.5))
x_pos = np.arange(len(cases))
ax3.bar(x_pos, bias_sq[order], width=0.7, color='#d62728',
        label='Bias² (model mis-specification)',
        alpha=0.85, edgecolor='k', linewidth=0.5)
ax3.bar(x_pos, variance[order], bottom=bias_sq[order], width=0.7,
        color='#1f77b4', label='Variance (window spread)',
        alpha=0.85, edgecolor='k', linewidth=0.5)
labels = [f"{cases[i].replace('Leak ','Leak_')}\n{ql_per_case[i]:.2f} l/s"
          for i in order]
ax3.set_xticks(x_pos)
ax3.set_xticklabels(labels, fontsize=8)
ax3.set_ylabel('Error budget [m²]', fontsize=12)
ax3.legend(loc='upper center', bbox_to_anchor=(0.5, 1.0),
           ncol=2, frameon=True, fontsize=10)
ax3.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig7.savefig('Figure_7.png')
generated_files.append('Figure_7.png')
plt.close(fig7)

# ─── FIGURE 8  (FIXED: legend and coverage stats non-overlapping) ─────────────
# Layout: legend → upper-left (sparse region), coverage text → lower-right
fig8, ax4 = plt.subplots(figsize=(8, 6))

ax4.plot([0, L_PIPE], [0, L_PIPE], 'k--', lw=0.8, alpha=0.6)

for i in range(len(cases)):
    t = y_true_case[i]; p = y_pred_pls[i]
    # 90% interval (thin, transparent)
    ax4.errorbar(t, p, yerr=[[p - lo90[i]], [hi90[i] - p]],
                 fmt='none', ecolor='#2ca02c', elinewidth=1.2,
                 capsize=0, alpha=0.35)
    # 80% interval (thick, opaque)
    ax4.errorbar(t, p, yerr=[[p - lo80[i]], [hi80[i] - p]],
                 fmt='none', ecolor='#2ca02c', elinewidth=2.5,
                 capsize=4, alpha=0.80)
    ax4.scatter(t, p, s=80, c='#2ca02c', marker='o',
                edgecolor='black', linewidth=0.7, zorder=4)

n_cov_90 = int(jackknife_plus_coverage(0.1, y_pred_pls, y_true_case)[0] * len(cases))
n_cov_80 = int(jackknife_plus_coverage(0.2, y_pred_pls, y_true_case)[0] * len(cases))

# Legend — upper-left (very little data there)
legend8 = [
    Line2D([0],[0], color='k', ls='--', lw=0.8, label='Ideal (y = x)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#2ca02c',
           markersize=9, markeredgecolor='k', label='Point prediction'),
    Line2D([0],[0], color='#2ca02c', lw=2.5, alpha=0.80,
           label='80% conformal interval'),
    Line2D([0],[0], color='#2ca02c', lw=1.2, alpha=0.35,
           label='90% conformal interval'),
]
ax4.legend(handles=legend8, loc='upper left', frameon=True,
           fontsize=10, borderpad=0.8)

# Coverage stats — lower-right corner (data-free region below the diagonal)
ax4.text(0.97, 0.05,
         f'90% nominal: {n_cov_90}/{len(cases)} covered\n'
         f'80% nominal: {n_cov_80}/{len(cases)} covered',
         transform=ax4.transAxes, fontsize=10,
         va='bottom', ha='right',
         bbox=dict(boxstyle='round', fc='white', ec='#2ca02c',
                   linewidth=1.2, alpha=0.95))

ax4.set_xlabel('True leak location [m]', fontsize=12)
ax4.set_ylabel('Predicted leak location [m]', fontsize=12)
ax4.set_xlim(0, L_PIPE)
ax4.set_ylim(-10, 75)
ax4.grid(alpha=0.3)
plt.tight_layout()
fig8.savefig('Figure_8.png')
generated_files.append('Figure_8.png')
plt.close(fig8)

# ==============================================================================
# STEP 8: AUTO-DOWNLOAD ALL FIGURES
# ==============================================================================
print(f"--- DOWNLOADING {len(generated_files)} FIGURES ---")
for img_file in generated_files:
    print(f"  Downloading: {img_file}")
    files.download(img_file)

print("\n--- ANALYSIS AND DOWNLOADING COMPLETED SUCCESSFULLY ---")


--- STEP 1: SYSTEM INITIALIZATION, CLEANUP, AND FILE UPLOAD ---
WORKSPACE CLEANED. PLEASE SELECT YOUR 9 .NPY FILES:


Saving Leak 1_15m.npy to Leak 1_15m.npy
Saving Leak 1_30m.npy to Leak 1_30m.npy
Saving Leak 1_50m.npy to Leak 1_50m.npy
Saving Leak 2_15m.npy to Leak 2_15m.npy
Saving Leak 2_30m.npy to Leak 2_30m.npy
Saving Leak 2_50m.npy to Leak 2_50m.npy
Saving Leak 3_15m.npy to Leak 3_15m.npy
Saving Leak 3_30m.npy to Leak 3_30m.npy
Saving Leak 3_50m.npy to Leak 3_50m.npy
SUCCESS: 9 files successfully loaded. Commencing analysis...

--- STEP 2: PERFORMING BOOTSTRAP ANALYSIS ON ANALYTIC BASELINE ---
--- STEP 3: EXTRACTING 33-DIMENSIONAL MULTI-SCALE FEATURES ---
--- STEP 4: TRAINING PLS-K3 MODEL VIA LEAVE-ONE-GROUP-OUT CV ---
   => Analytic Baseline MAE : 14.84 m
   => Hybrid PLS-k3 MAE     : 2.64 m  (Improvement: 5.6x)

--- STEP 5: CALCULATING JACKKNIFE+ CONFORMAL INTERVALS ---
--- STEP 6: GENERATING, SAVING, AND DOWNLOADING ALL 8 FIGURES ---
--- DOWNLOADING 8 FIGURES ---
  Downloading: Figure_1.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Downloading: Figure_2.png


...

<IPython.core.display.Javascript object>

  Downloading: Figure_8.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- ANALYSIS AND DOWNLOADING COMPLETED SUCCESSFULLY ---
